In [ ]:
# Uncomment kode di bawah ini jika terdapat masalah dengan Java di Windows

# import os
# import sys

# JAVA = r"C:\path\to\your\java\jdk" # Ganti dengan path JDK yang sesuai

# os.environ["JAVA_HOME"] = JAVA
# os.environ["PATH"] = os.path.join(JAVA, "bin") + os.path.pathsep + os.environ.get("PATH", "")
# os.environ["PYSPARK_PYTHON"] = sys.executable
# os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [ ]:
%pip install -r requirements.txt
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, length

spark = SparkSession.builder \
    .appName("AmazonReviewsPolarity") \
    .config("spark.driver.memory", "4g") \
    .master("local[*]") \
    .getOrCreate()

df = spark.read.csv("../datasets/test.csv", header=False, inferSchema=True)
df = df.toDF("label", "title", "text")

sc = spark.sparkContext

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
t0_start = time.perf_counter()
rdd_raw = sc.textFile("../datasets/test.csv")
rdd_parsed = rdd_raw.map(lambda line: line.split("|")).map(
    lambda cols: (
        int(cols[0]) if cols[0].isdigit() else None,
        cols[1] if len(cols) > 1 else "",
        cols[2] if len(cols) > 2 else "",
    )
)

count_rdd = rdd_parsed.count()

t0_end = time.perf_counter()
waktu_rdd = t0_end - t0_start
print(
    f"Operasi 1 (RDD API PySpark)               : {waktu_rdd:.4f} detik | Total"
    f" Baris: {count_rdd}"
)

t1_start = time.perf_counter()

df_infer = spark.read.csv(
    "../datasets/test.csv", header=False, inferSchema=True
).toDF("label", "title", "text")

count_df = df_infer.count()

t1_end = time.perf_counter()
waktu_df = t1_end - t1_start
print(
    f"Operasi 2 (DataFrame PySpark)  : {waktu_df:.4f} detik | Total"
    f" Baris: {count_df}"
)

selisih = abs(waktu_rdd - waktu_df)
print(f"\nSelisih waktu eksekusi: {selisih:.2f} detik.")

Operasi 1 (RDD API PySpark): 9.3635 detik | Total Baris: 400000
Operasi 2 (DataFrame PySpark): 1.6374 detik | Total Baris: 400000

Selisih waktu eksekusi: 7.73 detik.


In [15]:
print("Skema Data: ")
df.printSchema()
print(f"Jumlah Baris: {df.count()}")
print(f"Jumlah Partisi: {df.rdd.getNumPartitions()}")
print(f"Contoh Data: ")
df.show(5)

Skema Data: 
root
 |-- label: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- text: string (nullable = true)

Jumlah Baris: 399999
Jumlah Partisi: 8
Contoh Data: 
+-----+--------------------+--------------------+
|label|               title|                text|
+-----+--------------------+--------------------+
|    2|One of the best g...|Despite the fact ...|
|    1|Batteries died wi...|I bought this cha...|
|    2|works fine, but M...|Check out Maha En...|
|    2|Great for the non...|Reviewed quite a ...|
|    1|DVD Player crappe...|I also began havi...|
+-----+--------------------+--------------------+
only showing top 5 rows


In [28]:
print("Contoh data dengan label 1 (Negatif):")
df.filter(df.label == 1).show(5)

Contoh data dengan label 1 (Negatif):
+-----+--------------------+--------------------+--------+
|label|               title|                text|sentimen|
+-----+--------------------+--------------------+--------+
|    1|Batteries died wi...|I bought this cha...| Negatif|
|    1|DVD Player crappe...|I also began havi...| Negatif|
|    1|      Incorrect Disc|I love the style ...| Negatif|
|    1|DVD menu select p...|I cannot scroll t...| Negatif|
|    1|"Not an ""ultimat...|Firstly,I enjoyed...| Negatif|
+-----+--------------------+--------------------+--------+
only showing top 5 rows


In [29]:
print("Urutan 5 data pertama berdasarkan label secara menurun:")
df.orderBy(df.label.desc()).show(5)

Urutan 5 data pertama berdasarkan label secara menurun:
+-----+--------------------+--------------------+--------+
|label|               title|                text|sentimen|
+-----+--------------------+--------------------+--------+
|    2|One of the best g...|Despite the fact ...| Positif|
|    2|works fine, but M...|Check out Maha En...| Positif|
|    2|Great for the non...|Reviewed quite a ...| Positif|
|    2|Unique Weird Orie...|"Exotic tales of ...| Positif|
|    2|Great book for tr...|I currently live ...| Positif|
+-----+--------------------+--------------------+--------+
only showing top 5 rows


In [30]:
print("Jumlah data untuk setiap label:")
df.groupBy("label").count().show()

Jumlah data untuk setiap label:
+-----+------+
|label| count|
+-----+------+
|    1|200000|
|    2|199999|
+-----+------+



In [31]:
print("Penambahan kolom baru 'panjang_teks' yang berisi panjang teks dari kolom 'text' dan 'title':")
df.withColumn("panjang_teks", length(col("text")) + length(col("title"))).show(5)

Penambahan kolom baru 'panjang_teks' yang berisi panjang teks dari kolom 'text' dan 'title':
+-----+--------------------+--------------------+--------+------------+
|label|               title|                text|sentimen|panjang_teks|
+-----+--------------------+--------------------+--------+------------+
|    1|Batteries died wi...|I bought this cha...| Negatif|         333|
|    1|DVD Player crappe...|I also began havi...| Negatif|         378|
|    1|      Incorrect Disc|I love the style ...| Negatif|         406|
|    1|DVD menu select p...|I cannot scroll t...| Negatif|         251|
|    1|"Not an ""ultimat...|Firstly,I enjoyed...| Negatif|         883|
+-----+--------------------+--------------------+--------+------------+
only showing top 5 rows


In [32]:
print("Hasil penggabungan DataFrame dengan DataFrame referensi untuk menambahkan kolom sentimen:")
df_ref = spark.createDataFrame([(1, "Negatif"), (2, "Positif")], ["label", "sentimen"])
df.join(df_ref, "label").show(5)

Hasil penggabungan DataFrame dengan DataFrame referensi untuk menambahkan kolom sentimen:
+-----+--------------------+--------------------+--------+--------+
|label|               title|                text|sentimen|sentimen|
+-----+--------------------+--------------------+--------+--------+
|    1|Batteries died wi...|I bought this cha...| Negatif| Negatif|
|    1|DVD Player crappe...|I also began havi...| Negatif| Negatif|
|    1|      Incorrect Disc|I love the style ...| Negatif| Negatif|
|    1|DVD menu select p...|I cannot scroll t...| Negatif| Negatif|
|    1|"Not an ""ultimat...|Firstly,I enjoyed...| Negatif| Negatif|
+-----+--------------------+--------------------+--------+--------+
only showing top 5 rows
